In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict
from tqdm import tqdm
from itertools import product

from measurement_dist import measurement_dist
from pdg_data import load_pdg_data

In [ ]:
# point matplotlib ticks inwards
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"
# add top and right ticks
plt.rcParams["xtick.top"] = True
plt.rcParams["ytick.right"] = True
# use tex
plt.rcParams["text.usetex"] = True
plt.rcParams["text.latex.preamble"] = r"""
        \usepackage{libertine}
        \usepackage[libertine]{newtxmath}
"""

In [ ]:
_, pdg2025_both_dfs, _, _ = load_pdg_data()
pair_dist = measurement_dist(pdg2025_both_dfs, ci=False, asym_error=['error_negative', 'error_positive'], return_samples=True)

In [ ]:
zs = pair_dist['zs'] # shape (n_pairs,)
pair_years = pair_dist['pair_years'] # shape (n_pairs, 2)
pair_weights = pair_dist['pair_weights']
quantity_idxs_pair = pair_dist['quantity_idxs_pair']
zspace = pair_dist['zspace']
normdist = pair_dist['norm']
max_year = np.max(pair_years)
min_year = np.min(pair_years)
n_years = max_year-min_year+1
years = np.arange(min_year, max_year+1)

In [ ]:
masks = [pair_years[:,1] == i for i in years]
avg_zs = np.full((len(years), len(years)), np.nan)
avg_z2s = np.full((len(years), len(years)), np.nan)
avg_gt1sigs = np.full((len(years), len(years)), np.nan)
avg_gt2sigs = np.full((len(years), len(years)), np.nan)
from itertools import product
for i,j in product(range(len(years)), range(len(years))):

    mask = (pair_years[:,0] == years[i]) & (pair_years[:,1] == years[j])
    zs_mask = zs[mask]
    if len(zs_mask) == 0:
        avg_zs[i,j] = np.nan
        avg_z2s[i,j] = np.nan
        avg_gt1sigs[i,j] = np.nan
        avg_gt2sigs[i,j] = np.nan
        continue

    quantity_idx_to_count = dict(zip(*np.unique(quantity_idxs_pair[mask], return_counts=True)))
    pair_weights_mask = 1/np.array([quantity_idx_to_count[idx] for idx in quantity_idxs_pair[mask]])

    pair_weights_norm = pair_weights_mask/np.sum(pair_weights_mask)

    avg_zs[i,j] = np.sum(zs_mask * pair_weights_norm)
    avg_z2s[i,j] = np.sum(zs_mask**2 * pair_weights_norm)
    avg_gt1sigs[i,j] = np.sum((zs_mask > 1) * pair_weights_norm)
    avg_gt2sigs[i,j] = np.sum((zs_mask > 2) * pair_weights_norm)

In [ ]:
# Use TwoSlopeNorm
from matplotlib.colors import TwoSlopeNorm
div_cmap = plt.get_cmap('coolwarm')

year_grid = np.arange(min_year, max_year+1)+0.5
X, Y = np.meshgrid(year_grid, year_grid)

fig, axs = plt.subplots(1, 2, figsize=(8, 3))

# Left: avg_zs
# Use diverging colormap centered at sqrt(2/pi)

center = np.sqrt(2/np.pi)
color_norm = TwoSlopeNorm(vmin=0, vcenter=center, vmax=2)
pc1 = axs[0].pcolormesh(X, Y, avg_zs, cmap=div_cmap, norm=color_norm, shading='nearest')
cbar1 = fig.colorbar(pc1, ax=axs[0], label=r'Average $|z_{ij}|$', extend='max')
cbar1.ax.set_yscale('linear')
cbar1.ax.axhline(center, color='red')

# Right: avg_z2s
center = 1
color_norm = TwoSlopeNorm(vmin=0, vcenter=center, vmax=4)
pc2 = axs[1].pcolormesh(X, Y, avg_z2s, cmap=div_cmap, norm=color_norm, shading='nearest')
cbar2 = fig.colorbar(pc2, ax=axs[1], label=r'Average $z_{ij}^2$', extend='max')
cbar2.ax.set_yscale('linear')
cbar2.ax.axhline(center, color='red')

for ax in axs:
    ax.axhline(1990, color='black', linestyle='dashed', linewidth=1)
    ax.axvline(1990, color='black', linestyle='dashed', linewidth=1)
    # ax.axvline(2020, linewidth=1) # for checking tick alignment
    # ax.axvline(2025, linewidth=1)
    ax.set_xlabel('Year of later measurement')
    ax.set_ylabel('Year of earlier measurement')
    # ax.set_xticks(ticks, minor=True)
    # ax.set_yticks(ticks, minor=True)
    # force the x and y ticks to be the same
    ticks = ax.get_xticks()[1:-1]
    ax.set_yticks(ticks)
    ax.set_xticks(ticks)
    # add minor ticks


plt.tight_layout()
plt.savefig('../figs/blinding/yearbin2d.pdf', bbox_inches='tight')
plt.show()

In [ ]:
masks = [pair_years[:,1] == i for i in years]
avg_zs = []
avg_z2s = []
avg_gt1sigs = []
avg_gt2sigs = []
for i, mask in enumerate(masks):
    
    zs_mask = zs[mask]
    if len(zs_mask) == 0:
        avg_zs.append(np.nan)
        avg_z2s.append(np.nan)
        avg_gt1sigs.append(np.nan)
        avg_gt2sigs.append(np.nan)
        continue

    quantity_idx_to_count = dict(zip(*np.unique(quantity_idxs_pair[mask], return_counts=True)))
    pair_weights_mask = 1/np.array([quantity_idx_to_count[idx] for idx in quantity_idxs_pair[mask]])

    pair_weights_norm = pair_weights_mask/np.sum(pair_weights_mask)

    avg_zs.append(np.sum(zs_mask * pair_weights_norm))
    avg_z2s.append(np.sum(zs_mask**2 * pair_weights_norm))
    avg_gt1sigs.append(np.sum((zs_mask > 1) * pair_weights_norm))
    avg_gt2sigs.append(np.sum((zs_mask > 2) * pair_weights_norm))

fig, axes = plt.subplots(1, 3, figsize=(8, 3))
axs = axes.flatten()

linewidth = 1

axs[0].plot(years, avg_zs, color='black', linewidth=linewidth)
axs[0].set_xlabel('Year of later measurement')
axs[0].axhline(np.sqrt(2/np.pi), color='red', linewidth=linewidth)
axs[0].set_title(r'Average $|z_{ij}|$ vs Year')
axs[0].set_ylabel(r'Average $|z_{ij}|$')

axs[1].plot(years, avg_z2s, color='black', linewidth=linewidth)
axs[1].set_xlabel('Year of later measurement')
axs[1].axhline(1, color='red', linewidth=linewidth)
axs[1].set_title(r'Average $z_{ij}^2$ vs Year')
axs[1].set_ylabel(r'Average $z_{ij}^2$')

axs[2].plot(years, avg_gt1sigs, color='black', linewidth=linewidth)
axs[2].set_xlabel('Year of later measurement')
axs[2].axhline(1-0.6827, color='red', linewidth=linewidth)
axs[2].set_title(r'Fraction $|z_{ij}| > 1$ vs Year')
axs[2].set_ylabel(r'Fraction $|z_{ij}| > 1$')

plt.tight_layout()
plt.savefig('../figs/blinding/yearbin1d.pdf', bbox_inches='tight')

plt.show()

In [ ]:
def blinding_before_after(blinding_year, pair_years, quantity_idxs_pair, zs, boot=False, permute=False, only_quantities_in_both=True):
    mask0 = np.all(pair_years <= blinding_year, axis=1)
    mask1 = np.all(pair_years > blinding_year, axis=1)

    assert np.sum(mask0 & mask1) == 0


    if only_quantities_in_both:
        mask2 = np.zeros_like(mask0)

        mask0_quantities = np.unique(quantity_idxs_pair[mask0])
        mask1_quantities = np.unique(quantity_idxs_pair[mask1])
        
        for quantity in np.unique(quantity_idxs_pair):
            if quantity in mask0_quantities and quantity in mask1_quantities:
                mask2 = mask2 | (quantity_idxs_pair == quantity)
        mask0 = mask0 & mask2
        mask1 = mask1 & mask2

    avg_zs = []
    avg_z2s = []
    avg_gt1sigs = []
    avg_gt2sigs = []

    if boot:
        quantities = np.unique(quantity_idxs_pair)
        # sample of quantities
        quantity_idxs_sample = np.random.choice(quantities, size=len(quantities), replace=True)
        # count how many times each quantity appears
        unique, counts = np.unique(quantity_idxs_sample, return_counts=True)
        # map the quantity indices to the counts
        boot_quantity_count = defaultdict(int)
        boot_quantity_count.update(dict(zip(unique, counts)))

    if permute:
        mask0or1 = mask0 | mask1
        n0 = np.sum(mask0)
        n1 = np.sum(mask1)
        n_total = np.sum(n0+n1)

        # array with length of mask0or1 with n0 1s
        n0_bool = np.zeros(n_total, dtype=bool)
        n0_bool[:n0] = True
        np.random.shuffle(n0_bool)
        # print(n0_bool)

        mask0_p = np.zeros_like(mask0)
        mask1_p = np.zeros_like(mask1)
        mask0_p[mask0or1] = n0_bool
        mask1_p[mask0or1] = ~n0_bool

        assert np.sum(mask0_p) == n0, f'{np.sum(mask0_p)} != {n0}'
        assert np.sum(mask1_p) == n1

        mask0 = mask0_p
        mask1 = mask1_p


    for i, mask in enumerate([mask0, mask1]):
        
        zs_mask = zs[mask]

        quantity_idx_to_count = dict(zip(*np.unique(quantity_idxs_pair[mask], return_counts=True)))
        pair_weights_mask = 1/np.array([quantity_idx_to_count[idx] for idx in quantity_idxs_pair[mask]])
        if boot:
            pair_weights_mask = pair_weights_mask * np.array([boot_quantity_count[idx] for idx in quantity_idxs_pair[mask]])

        pair_weights_norm = pair_weights_mask/np.sum(pair_weights_mask)

        avg_zs.append(np.sum(zs_mask * pair_weights_norm))
        avg_z2s.append(np.sum(zs_mask**2 * pair_weights_norm))
        avg_gt1sigs.append(np.sum((zs_mask > 1) * pair_weights_norm))
        avg_gt2sigs.append(np.sum((zs_mask > 2) * pair_weights_norm))

    return np.array([avg_zs, avg_z2s, avg_gt1sigs, avg_gt2sigs])


In [ ]:
results_center = blinding_before_after(1990, pair_years, quantity_idxs_pair, zs, only_quantities_in_both=True)
results_center

In [ ]:

B = 4000
results_b = np.full((*results_center.shape, B), np.nan)
for b in tqdm(range(B)):
    results_b[:,:,b] = blinding_before_after(1990, pair_years, quantity_idxs_pair, zs, boot=True, only_quantities_in_both=True)

In [ ]:
# P = 4000
# results_p = np.full((*results_center.shape, P), np.nan)
# for p in tqdm(range(P)):
#     results_p[:,:,p] = blinding_before_after(1990, pair_years, quantity_idxs_pair, zs, permute=True, only_quantities_in_both=True)

In [ ]:
quantiles = np.quantile(results_b, [0.025, 0.975], axis=2)
print(quantiles.shape)
results_lower = 2*results_center - quantiles[1,:,:]
results_upper = 2*results_center - quantiles[0,:,:]
print(results_lower)
print(results_upper)

In [ ]:
diffs_center = results_center[:,1] - results_center[:,0]
diffs_b = results_b[:,1,:] - results_b[:,0,:]
quantiles = np.quantile(diffs_b, [0.025, 0.975], axis=1)
diffs_lower = 2*diffs_center - quantiles[1,:]
diffs_upper = 2*diffs_center - quantiles[0,:]
print(diffs_center)
print(diffs_lower)
print(diffs_upper)

In [ ]:
# permutation test doesn't work well; just switching around the
# pairs doesn't change the differences much; bootstrapping quantities
# changes them a lot more and is more realistic.image.png

# diffs_p = results_p[:,1,:] - results_p[:,0,:]
# perm_p_values = np.full(len(diffs_center), np.nan)
# for i in range(len(diffs_center)):
#     perm_p_values[i] = np.sum(diffs_p[i,:] >= diffs_center[i])/P
# print(perm_p_values)

In [ ]:
results_error_n = results_center - results_lower
results_error_p = results_upper - results_center
diffs_error_n = diffs_center - diffs_lower
diffs_error_p = diffs_upper - diffs_center

def format_asym(val, err_p, err_n, fmt=".3f", combine_if_equal=True):
    err_p_str = f"{err_p:{fmt}}"
    err_n_str = f"{err_n:{fmt}}"
    val_str = f"${val:{fmt}}"
    if err_p_str == err_n_str and combine_if_equal:
        return (
            val_str +
            rf"\pm{{{err_p_str}}}$"
        )
    else:
        return (
            val_str +
            r"^{+" + err_p_str + r"}" +
            r"_{-" + err_n_str + r"}$"
        )

rows = []

n = results_center.shape[0]
from scipy.stats import norm
for i in range(n):
    row_label = [r"Average $|z_{ij}|$", r"Average $z_{ij}^2$", r"Fraction $|z_{ij}| > 1$", r"Fraction $|z_{ij}| > 2$"][i]
    expectation = [f"{np.sqrt(2/np.pi):.3f}", "1", f"{2*norm.cdf(-1):.4f}", f"{2*norm.cdf(-2):.4f}"][i]
    col0 = format_asym(
        results_center[i, 0],
        results_error_p[i, 0],
        results_error_n[i, 0],
    )

    col1 = format_asym(
        results_center[i, 1],
        results_error_p[i, 1],
        results_error_n[i, 1],
    )

    col2 = format_asym(
        diffs_center[i],
        diffs_error_p[i],
        diffs_error_n[i],
    )

    rows.append(f"{row_label} & {expectation} & {col0} & {col1} & {col2} \\\\")

latex_table = (
    "\\begin{tabular}{l|rrr|r}\n"
    "\\hline\n"
    r"Statistic & Expectation & $<1990$ & $\geq 1990$ & Difference \\"
    "\\hline\n"
    + "\n".join(rows) +
    "\n\\hline\n"
    "\\end{tabular}"
)

print(latex_table)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3))

mask0 = np.all(pair_years <= 1990, axis=1)
mask1 = np.all(pair_years > 1990, axis=1)

# filter for only quantities which have a pair in both
mask2 = np.zeros_like(mask0)

mask0_quantities = np.unique(quantity_idxs_pair[mask0])
mask1_quantities = np.unique(quantity_idxs_pair[mask1])

for quantity in np.unique(quantity_idxs_pair):
    if quantity in mask0_quantities and quantity in mask1_quantities:
        mask2 = mask2 | (quantity_idxs_pair == quantity)
mask0 = mask0 & mask2
mask1 = mask1 & mask2

colors = ['grey', 'black']
labels = [r'$\leq 1990$', r'$> 1990$']
for i, mask in enumerate([mask0, mask1]):
    
    zs_mask = zs[mask]

    quantity_idx_to_count = dict(zip(*np.unique(quantity_idxs_pair[mask], return_counts=True)))
    pair_weights_mask = 1/np.array([quantity_idx_to_count[idx] for idx in quantity_idxs_pair[mask]])

    pair_weights_norm = pair_weights_mask/np.sum(pair_weights_mask)

    for ax in axs:
        ax.plot(zspace, [np.sum((zs_mask > z)*pair_weights_norm) for z in zspace], color=colors[i], linewidth=1.5, label=labels[i])
for ax in axs:
    ax.plot(zspace, normdist, color='black', linestyle='dashed', linewidth=2, label='$|N(0,1)|$')
    ax.set_xlabel('$z$')
    ax.set_ylabel(r'$P(|Z|>z)$')

axs[0].set_xlim(0,2)
axs[0].set_ylim(4e-2,1)
axs[1].set_xlim(0,5)
axs[1].set_ylim(1e-4, 1)
axs[1].set_yscale('log')
axs[0].set_yscale('log')
axs[1].legend(frameon=False)

# fetch table number from the aux file for the technical note
import re
with open("../technote/main.aux") as f:
    aux = f.read()
match = re.search(r'\\newlabel{tab:blinding-quadrants}{{([^}]*)}', aux)
table_number = match.group(1)
print(table_number)

# annotate axs[0] with an arrow pointing to (1, 1-0.6827) and some text
axs[0].annotate('', xy=(1, 1-0.6827), xytext=(1.2, 0.5),
                arrowprops=dict(arrowstyle='->', color='black'))
axs[0].annotate('', xy=(2, 0.09), xytext=(1.6, 0.5),
                arrowprops=dict(arrowstyle='->', color='black'))
axs[0].text(1.4, 0.5, rf'See Table {table_number}', ha='center', va='bottom', fontsize=10)

# axs[0].axvline(1, linestyle=':', color='black', linewidth=1)

plt.tight_layout()

rect, lines = axs[1].indicate_inset_zoom(axs[0])
for line in lines:
    line.set_visible(True)
    line.set_zorder(10)
axs[0].set_zorder(100)

plt.savefig('../figs/blinding/beforeafterdist.pdf', bbox_inches='tight')
plt.show()